# Teksta apstr?de ar Python

Teksta apstr?de parasti s?kas ar neapstr?d?tu tekstu, kuru vajag apskat?t, izt?r?t, analiz?t un eksport?t. Tipiski uzdevumi ietver rindu kvalit?tes p?rbaudi, re?istra un atstarpju normaliz??anu, pieturz?mju vai stopwords no?em?anu, bie??ko v?rdu skait??anu un da??ji struktur?ta teksta p?rveido?anu tabul? t?l?kai anal?zei.

?is notebook seko ?iem so?iem ar se?iem piem?riem, kas piel?goti no `scripts/day2` materi?liem. Katr? sada?? ir uzdevuma skaidrojums, atk?rtoti lietojami pal?gbloki un viena wrapper function izsaukums beig?s, lai notebook var?tu palaist no s?kuma l?dz beig?m ar **Run All**.

## Neapstr?d?ta teksta faila p?rbaude

Teksta apstr?des pl?smu ir v?rts s?kt ar p?rbaudi. Pirms kaut ko main?m, ir noder?gi redz?t, cik rindas fail? ir kop?, kuras rindas ir tuk?as, vai tekst? negaid?ti par?d?s cipari, vai atstarpes ir nekonsekventas un vai da?as atbildes nedubl?jas.

In [ ]:
from collections import Counter
from pathlib import Path
import csv
import re
import string


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / '.venv').exists() and (candidate / 'data' / 'day2').exists():
            return candidate
        if (candidate / 'data' / 'day2').exists():
            return candidate
    raise FileNotFoundError('Neizdev?s atrast projekta sakni, kur? atrodas data/day2.')


PROJECT_ROOT = find_project_root()
DAY2_DATA_DIR = PROJECT_ROOT / 'data' / 'day2'
RAW_PATH = DAY2_DATA_DIR / 'responses_raw.txt'
STOPWORDS_PATH = DAY2_DATA_DIR / 'stopwords_lv.txt'
GENERATED_CLEANED_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses.txt'
GENERATED_FREQ_PATH = DAY2_DATA_DIR / 'generated_word_frequencies.txt'
GENERATED_CLEANED_V2_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses_v2.txt'
GENERATED_FREQ_V2_PATH = DAY2_DATA_DIR / 'generated_word_frequencies_v2.txt'
GENERATED_CLEANED_FINAL_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses_final.txt'
GENERATED_FREQ_FINAL_PATH = DAY2_DATA_DIR / 'generated_word_frequencies_final.txt'
SURVEYS_FOLDER = DAY2_DATA_DIR / 'surveys'
SURVEY_OUTPUT_PATH = SURVEYS_FOLDER / 'survey_responses_summary.csv'

print(f'Projekta sakne: {PROJECT_ROOT}')
print(f'2. dienas datu mape: {DAY2_DATA_DIR}')

In [ ]:
def read_text_file(path: Path, encoding: str = 'utf-8') -> str:
    '''Nolasa teksta failu un atgrie? visu saturu k? vienu virkni.'''
    with path.open('r', encoding=encoding) as file:
        return file.read()


def split_into_lines(text: str) -> list[str]:
    '''Sadala tekstu rind?s, nesaglab?jot rindas beigu simbolus.'''
    return text.splitlines()


def find_blank_lines(lines: list[str]) -> list[int]:
    '''Atgrie? 1-b?z?tus rindu numurus tuk??m vai tikai atstarpes saturo??m rind?m.'''
    return [i for i, line in enumerate(lines, start=1) if line.strip() == '']


def find_digit_lines(lines: list[str]) -> list[int]:
    '''Atgrie? 1-b?z?tus rindu numurus rind?m, kur?s ir vismaz viens cipars.'''
    return [i for i, line in enumerate(lines, start=1) if any(ch.isdigit() for ch in line)]


def find_lines_with_extra_spaces(lines: list[str]) -> list[int]:
    '''Atgrie? 1-b?z?tus rindu numurus rind?m ar dubultatstarp?m.'''
    return [i for i, line in enumerate(lines, start=1) if '  ' in line]


def find_duplicate_lines(lines: list[str]) -> list[tuple[str, int]]:
    '''Atgrie? dubl?tas netuk??s rindas un to par?d??an?s rei?u skaitu.'''
    counts = Counter(line.strip() for line in lines if line.strip() != '')
    return [(line, count) for line, count in counts.items() if count > 1]


def print_inspection_report(lines: list[str]) -> None:
    '''Izdruk? kompaktu neapstr?d?t? teksta p?rskatu.'''
    blank_lines = find_blank_lines(lines)
    digit_lines = find_digit_lines(lines)
    extra_space_lines = find_lines_with_extra_spaces(lines)
    duplicate_lines = find_duplicate_lines(lines)

    print('Neapstr?d?ta teksta p?rskats')
    print(f'- Rindu skaits kop?: {len(lines)}')
    print(f'- Tuk??s rindas: {blank_lines}')
    print(f'- Rindas ar cipariem: {digit_lines}')
    print(f'- Rindas ar atk?rtot?m atstarp?m: {extra_space_lines}')
    print(f'- Dubl?tu netuk?u rindu skaits: {len(duplicate_lines)}')

    print('\nPirm?s 5 rindas')
    for i, line in enumerate(lines[:5], start=1):
        print(f'{i}: {line}')

    if duplicate_lines:
        print('\nDubl?tu rindu piem?ri')
        for line, count in duplicate_lines[:3]:
            print(f'{count}x | {line}')


In [ ]:
def inspect_text_file(path: Path = RAW_PATH) -> dict[str, object]:
    '''Izpilda neapstr?d?t? teksta p?rbaudes pl?smu ?ai notebook sada?ai.'''
    text = read_text_file(path)
    lines = split_into_lines(text)
    summary = {
        'path': str(path),
        'total_lines': len(lines),
        'blank_lines': find_blank_lines(lines),
        'digit_lines': find_digit_lines(lines),
        'extra_space_lines': find_lines_with_extra_spaces(lines),
        'duplicate_lines': find_duplicate_lines(lines),
    }
    print_inspection_report(lines)
    return summary


inspection_summary = inspect_text_file()
inspection_summary

## Teksta rindu t?r??ana

P?c p?rbaudes n?kamais uzdevums ir normaliz??ana. T? parasti ietver teksta p?rveido?anu uz mazajiem burtiem, lieko atstarpju no?em?anu, pieturz?mju iz?em?anu, atk?rtotu atstarpju saspie?anu un to rindu atme?anu, kuras p?c t?r??anas k??st tuk?as.

In [ ]:
def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Nolasa teksta failu un atgrie? t? rindas bez rindas beigu simboliem.'''
    with path.open('r', encoding=encoding) as file:
        return file.read().splitlines()


def clean_line(line: str) -> str:
    '''Normaliz? vienu teksta rindu t?r?k? form?.'''
    line = line.lower().strip()
    line = line.translate(str.maketrans('', '', string.punctuation))
    line = ' '.join(line.split())
    return line


def clean_lines(lines: list[str]) -> list[str]:
    '''Izt?ra visas rindas un atmet tuk?os rezult?tus.'''
    cleaned = []
    for line in lines:
        cleaned_line = clean_line(line)
        if cleaned_line != '':
            cleaned.append(cleaned_line)
    return cleaned


def write_lines(path: Path, lines: list[str], encoding: str = 'utf-8') -> None:
    '''Saglab? rindas UTF-8 teksta fail?, pa vienai rindai katr? izvades rind?.'''
    with path.open('w', encoding=encoding) as file:
        for line in lines:
            file.write(line + '\n')


def compare_raw_and_cleaned(raw_lines: list[str], cleaned_lines: list[str], limit: int = 5) -> None:
    '''Izdruk? nelielu sal?dzin?jumu starp neapstr?d?t?m un t?r?t?m rind?m.'''
    print('Neapstr?d?tu un t?r?tu rindu paraugs')
    for raw, cleaned in zip(raw_lines[:limit], cleaned_lines[:limit]):
        print(f'RAW   : {raw!r}')
        print(f'CLEAN : {cleaned!r}')
        print('---')


In [ ]:
def clean_text_lines(
    raw_path: Path = RAW_PATH,
    output_path: Path = GENERATED_CLEANED_PATH,
) -> list[str]:
    '''Izpilda rindu t?r??anas pl?smu un saglab? izt?r?to tekstu.'''
    raw_lines = read_lines(raw_path)
    cleaned_lines = clean_lines(raw_lines)
    write_lines(output_path, cleaned_lines)

    print(f'Izt?r?t?s rindas saglab?tas: {output_path}')
    print(f'Neapstr?d?to rindu skaits: {len(raw_lines)}')
    print(f'Izt?r?to rindu skaits: {len(cleaned_lines)}')
    compare_raw_and_cleaned(raw_lines, cleaned_lines)
    return cleaned_lines


cleaned_lines_output = clean_text_lines()
cleaned_lines_output[:5]

## V?rdu bie?uma p?rskata izveide

Kad teksta rindas ir izt?r?tas, t?s var sadal?t v?rdos. Bie?s anal?zes solis ir stopwords no?em?ana, atliku?o v?rdu bie?uma saskait??ana, rezult?tu sak?rto?ana un bie?uma p?rskata saglab??ana v?l?kai apskatei.

In [ ]:
def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Nolasa teksta failu un atgrie? rindas bez rindas beigu simboliem.'''
    with path.open('r', encoding=encoding) as file:
        return file.read().splitlines()


def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    '''Iel?d? stopwords no teksta faila kop?.'''
    lines = read_lines(path, encoding=encoding)
    return {line.strip() for line in lines if line.strip() != ''}


def tokenize_lines(lines: list[str]) -> list[str]:
    '''Sadala izt?r?t?s rindas plakan? token sarakst?.'''
    tokens = []
    for line in lines:
        tokens.extend(line.split())
    return tokens


def remove_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
    '''Iz?em stopwords no token saraksta.'''
    return [token for token in tokens if token not in stopwords]


def count_words(tokens: list[str]) -> dict[str, int]:
    '''Saskaita token bie?umus ar v?rdn?cas pal?dz?bu.'''
    counts: dict[str, int] = {}
    for token in tokens:
        counts[token] = counts.get(token, 0) + 1
    return counts


def sort_word_counts(word_counts: dict[str, int]) -> list[tuple[str, int]]:
    '''Sak?rto v?rdu bie?umus dilsto?i p?c skaita un p?c tam alfab?tiski.'''
    return sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))


def write_word_frequencies(
    path: Path,
    word_counts: list[tuple[str, int]],
    encoding: str = 'utf-8',
) -> None:
    '''Saglab? v?rdu bie?umus k? tab-atdal?tas rindas.'''
    with path.open('w', encoding=encoding) as file:
        for word, count in word_counts:
            file.write(f'{word}\t{count}\n')


In [ ]:
def build_word_frequency_report(
    cleaned_path: Path = GENERATED_CLEANED_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    output_path: Path = GENERATED_FREQ_PATH,
) -> list[tuple[str, int]]:
    '''Izveido un saglab? v?rdu bie?uma p?rskatu no izt?r?ta teksta.'''
    cleaned_lines = read_lines(cleaned_path)
    stopwords = load_stopwords(stopwords_path)
    tokens = tokenize_lines(cleaned_lines)
    filtered_tokens = remove_stopwords(tokens, stopwords)
    word_counts = count_words(filtered_tokens)
    sorted_counts = sort_word_counts(word_counts)
    write_word_frequencies(output_path, sorted_counts)

    print(f'Bie?uma p?rskats saglab?ts: {output_path}')
    print(f'Token skaits: {len(tokens)}')
    print(f'Filtr?to token skaits: {len(filtered_tokens)}')
    print('Top 10 v?rdi:')
    for word, count in sorted_counts[:10]:
        print(f'{word}\t{count}')
    return sorted_counts


word_frequency_report = build_word_frequency_report()
word_frequency_report[:10]

## Refaktor??ana ar comprehensions un generators

To pa?u pl?smu var uzrakst?t kompakt?k, izmantojot comprehensions un generators. ?? sada?a saglab? tos pa?us so?us, bet p?rraksta da?u no t?r??anas un skait??anas pipeline ?s?k? intermediate Python stil?.

In [ ]:
def clean_line(line: str) -> str:
    '''Normaliz? vienu teksta rindu.'''
    line = line.lower().strip()
    line = line.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(line.split())


def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Nolasa UTF-8 teksta failu rindu sarakst?.'''
    with path.open('r', encoding=encoding) as file:
        return file.read().splitlines()


def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    '''Iel?d? stopwords kop?.'''
    return {line.strip() for line in read_lines(path, encoding=encoding) if line.strip() != ''}


def cleaned_lines_from_file(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Nolasa un izt?ra rindas ar list comprehension pal?dz?bu.'''
    lines = read_lines(path, encoding=encoding)
    return [clean_line(line) for line in lines if clean_line(line) != '']


In [ ]:
def cleaned_line_generator(path: Path, encoding: str = 'utf-8'):
    '''Atgrie? izt?r?tas netuk?as rindas pa vienai ar generator pal?dz?bu.'''
    for line in read_lines(path, encoding=encoding):
        cleaned = clean_line(line)
        if cleaned != '':
            yield cleaned


def token_generator(lines: list[str]):
    '''Atgrie? token pa vienam no izt?r?taj?m rind?m.'''
    for line in lines:
        for token in line.split():
            yield token


def build_word_counts(tokens: list[str], stopwords: set[str]) -> dict[str, int]:
    '''Izveido bie?umu v?rdn?cu no filtr?tiem token.'''
    filtered_tokens = [token for token in tokens if token not in stopwords]
    unique_tokens = {token for token in filtered_tokens}
    return {token: filtered_tokens.count(token) for token in unique_tokens}


def summarize_counts(word_counts: dict[str, int], top_n: int = 10) -> list[tuple[str, int]]:
    '''Atgrie? top N v?rdu bie?umus.'''
    return sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))[:top_n]


def write_lines(path: Path, lines: list[str], encoding: str = 'utf-8') -> None:
    '''Saglab? parastas teksta rindas fail?.'''
    with path.open('w', encoding=encoding) as file:
        for line in lines:
            file.write(line + '\n')


def write_word_frequencies(path: Path, pairs: list[tuple[str, int]], encoding: str = 'utf-8') -> None:
    '''Saglab? v?rdu bie?uma p?rus fail?.'''
    with path.open('w', encoding=encoding) as file:
        for word, count in pairs:
            file.write(f'{word}\t{count}\n')


In [ ]:
def run_comprehension_pipeline(
    raw_path: Path = RAW_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    cleaned_output_path: Path = GENERATED_CLEANED_V2_PATH,
    frequency_output_path: Path = GENERATED_FREQ_V2_PATH,
) -> dict[str, object]:
    '''Izpilda teksta pipeline kompakt?k? intermediate Python stil?.'''
    stopwords = load_stopwords(stopwords_path)
    cleaned_lines = cleaned_lines_from_file(raw_path)
    tokens = list(token_generator(cleaned_lines))
    word_counts = build_word_counts(tokens, stopwords)
    top_words = summarize_counts(word_counts, top_n=10)

    write_lines(cleaned_output_path, cleaned_lines)
    write_word_frequencies(
        frequency_output_path,
        sorted(word_counts.items(), key=lambda item: (-item[1], item[0])),
    )

    print(f'Izt?r?t? izvade saglab?ta: {cleaned_output_path}')
    print(f'Bie?uma izvade saglab?ta: {frequency_output_path}')
    print(f'Izt?r?to rindu skaits: {len(cleaned_lines)}')
    print(f'Token skaits: {len(tokens)}')
    print('Top 10 v?rdi:')
    for word, count in top_words:
        print(f'{word}\t{count}')

    return {
        'cleaned_lines': cleaned_lines,
        'tokens': tokens,
        'word_counts': word_counts,
        'top_words': top_words,
    }


comprehension_pipeline_results = run_comprehension_pipeline()
comprehension_pipeline_results['top_words']

## Pilnas pl?smas iepako?ana klas?

Kad pipeline k??st liel?ks, ir noder?gi glab?t saist?tos datus un uzved?bu kop?. ?? sada?a visu pl?smu ieliek `TextCorpus` klas?, kas var iel?d?t failus, t?r?t rindas, veidot token, apr??in?t bie?umus, saglab?t rezult?tus un izveidot kopsavilkumu.

In [ ]:
def clean_line(line: str) -> str:
    '''Normaliz? vienu neapstr?d?ta teksta rindu.'''
    line = line.lower().strip()
    line = line.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(line.split())


class TextCorpus:
    '''Neliels atk?rtoti lietojams objekts teksta failu iel?dei, t?r??anai un anal?zei.'''

    def __init__(self, raw_path: Path, stopwords_path: Path, encoding: str = 'utf-8'):
        self.raw_path = raw_path
        self.stopwords_path = stopwords_path
        self.encoding = encoding
        self.raw_lines: list[str] = []
        self.stopwords: set[str] = set()
        self.cleaned_lines: list[str] = []
        self.tokens: list[str] = []

    def load(self) -> None:
        '''Iel?d? neapstr?d?t?s rindas un stopwords no diska.'''
        with self.raw_path.open('r', encoding=self.encoding) as file:
            self.raw_lines = file.read().splitlines()

        with self.stopwords_path.open('r', encoding=self.encoding) as file:
            self.stopwords = {line.strip() for line in file if line.strip() != ''}

    def clean(self) -> None:
        '''Izt?ra neapstr?d?t?s rindas un saglab? rezult?tu.'''
        self.cleaned_lines = [clean_line(line) for line in self.raw_lines if clean_line(line) != '']

    def tokenize(self) -> None:
        '''Sadala izt?r?t?s rindas token sarakst?.'''
        self.tokens = [token for line in self.cleaned_lines for token in line.split()]

    def filtered_tokens(self) -> list[str]:
        '''Atgrie? token sarakstu bez stopwords.'''
        return [token for token in self.tokens if token not in self.stopwords]

    def word_counts(self) -> dict[str, int]:
        '''Atgrie? filtr?to token bie?umus.'''
        counts: dict[str, int] = {}
        for token in self.filtered_tokens():
            counts[token] = counts.get(token, 0) + 1
        return counts

    def top_words(self, n: int = 10) -> list[tuple[str, int]]:
        '''Atgrie? top N v?rdus p?c bie?uma.'''
        counts = self.word_counts()
        return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:n]

    def save_cleaned(self, path: Path) -> None:
        '''Saglab? izt?r?t?s rindas teksta fail?.'''
        with path.open('w', encoding=self.encoding) as file:
            for line in self.cleaned_lines:
                file.write(line + '\n')

    def save_word_frequencies(self, path: Path) -> None:
        '''Saglab? sak?rtotus v?rdu bie?umus tab-atdal?t? teksta fail?.'''
        with path.open('w', encoding=self.encoding) as file:
            for word, count in sorted(self.word_counts().items(), key=lambda item: (-item[1], item[0])):
                file.write(f'{word}\t{count}\n')

    def summary(self) -> dict[str, int]:
        '''Atgrie? kompaktu korpusa skaitlisko kopsavilkumu.'''
        filtered = self.filtered_tokens()
        return {
            'raw_line_count': len(self.raw_lines),
            'cleaned_line_count': len(self.cleaned_lines),
            'token_count': len(self.tokens),
            'filtered_token_count': len(filtered),
            'unique_word_count': len(set(filtered)),
        }


In [ ]:
def run_text_corpus_pipeline(
    raw_path: Path = RAW_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    cleaned_output_path: Path = GENERATED_CLEANED_FINAL_PATH,
    frequency_output_path: Path = GENERATED_FREQ_FINAL_PATH,
) -> dict[str, object]:
    '''Izpilda pilno klases pieeju teksta apstr?des pl?smai.'''
    corpus = TextCorpus(raw_path, stopwords_path)
    corpus.load()
    corpus.clean()
    corpus.tokenize()
    corpus.save_cleaned(cleaned_output_path)
    corpus.save_word_frequencies(frequency_output_path)

    print(f'Izt?r?t? izvade saglab?ta: {cleaned_output_path}')
    print(f'Bie?uma izvade saglab?ta: {frequency_output_path}')

    print('\nKopsavilkums')
    for key, value in corpus.summary().items():
        print(f'- {key}: {value}')

    print('\nTop 10 v?rdi')
    for word, count in corpus.top_words(10):
        print(f'{word}\t{count}')

    return {
        'summary': corpus.summary(),
        'top_words': corpus.top_words(10),
        'corpus': corpus,
    }


text_corpus_results = run_text_corpus_pipeline()
text_corpus_results['summary']

## Aptaujas failu p?rveido?ana par CSV

Teksta apstr?de ir noder?ga ar? tad, ja ievade nav piln?gi br?v? form?, bet ir da??ji struktur?ta. ?aj? p?d?j? sada?? aptaujas teksta failu mape tiek pars?ta ar regular expressions, svar?g?s v?rt?bas tiek izvilktas, gar?kas atbildes tiek reduc?tas l?dz keywords un rezult?ti tiek saglab?ti CSV tabul?.

In [ ]:
DEFAULT_OUTPUT_NAME = 'survey_responses_summary.csv'

QUESTION_FIELD_MAP = {
    1: 'name',
    2: 'age_answer',
    3: 'gender',
    4: 'contact_answer',
    5: 'employment_answer',
    6: 'education_answer',
    7: 'household_answer',
    8: 'income_answer',
    9: 'economic_pressures_answer',
    10: 'drive_answer',
    11: 'bike_answer',
    12: 'inflation_impact_answer',
    13: 'job_finance_security_answer',
    14: 'riga_economy_view_answer',
    15: 'improvements_answer',
}

TEXT_FIELDS_FOR_KEYWORDS = [
    'employment_answer',
    'education_answer',
    'household_answer',
    'income_answer',
    'economic_pressures_answer',
    'inflation_impact_answer',
    'job_finance_security_answer',
    'riga_economy_view_answer',
    'improvements_answer',
]

CSV_COLUMNS = [
    'source_file',
    'name',
    'age',
    'age_answer',
    'gender',
    'email',
    'phone',
    'employment_type',
    'occupation',
    'years_in_profession',
    'employment_answer',
    'employment_answer_keywords',
    'education_answer',
    'education_answer_keywords',
    'household_answer',
    'household_answer_keywords',
    'income_min_eur',
    'income_max_eur',
    'income_answer',
    'income_answer_keywords',
    'economic_pressures_answer',
    'economic_pressures_answer_keywords',
    'drive_km_week',
    'bike_km_week',
    'inflation_impact_answer',
    'inflation_impact_answer_keywords',
    'job_finance_security_answer',
    'job_finance_security_answer_keywords',
    'riga_economy_view_answer',
    'riga_economy_view_answer_keywords',
    'improvements_answer',
    'improvements_answer_keywords',
]


In [ ]:
def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    '''Iel?d? stopwords no UTF-8 teksta faila.'''
    with path.open('r', encoding=encoding) as file:
        return {line.strip().lower() for line in file if line.strip() != ''}


def read_text(path: Path, encoding: str = 'utf-8') -> str:
    '''Nolasa UTF-8 teksta failu.'''
    with path.open('r', encoding=encoding) as file:
        return file.read()


def parse_survey_blocks(text: str) -> dict[int, str]:
    '''Izvelk numur?tus jaut?jumu-atbil?u blokus ar regex pal?dz?bu.'''
    pattern = re.compile(r'(?ms)^\s*(\d+)\.\s+.*?\n(.*?)(?=^\s*\d+\.\s+|\Z)')
    answers: dict[int, str] = {}

    for match in pattern.finditer(text):
        question_number = int(match.group(1))
        answer = match.group(2).strip()
        answers[question_number] = answer

    return answers


def normalize_text(text: str) -> str:
    '''P?rveido tekstu uz mazajiem burtiem un saglab? v?rdveid?gus token keyword ieguvei.'''
    lowered = text.lower()
    cleaned = re.sub(r'[^\w\s]', ' ', lowered, flags=re.UNICODE)
    return ' '.join(cleaned.split())


def keyword_text(text: str, stopwords: set[str]) -> str:
    '''Iz?em stopwords no teksta un atgrie? normaliz?tus keywords.'''
    tokens = normalize_text(text).split()
    filtered_tokens = [token for token in tokens if token not in stopwords]
    return ' '.join(filtered_tokens)


def compact_value(text: str) -> str:
    '''No?em liek?s atstarpes un beigu teikuma pieturz?mes.'''
    return text.strip().rstrip('.')


def first_int(text: str) -> int | None:
    '''Atgrie? pirmo vesel? skait?a v?rt?bu, kas atrasta tekst?.'''
    match = re.search(r'\d+', text)
    return int(match.group()) if match else None


def income_range(text: str) -> tuple[int | None, int | None]:
    '''Izvelk minim?lo un maksim?lo ien?kumu v?rt?bu no latvie?u aptaujas teksta.'''
    numbers = [int(number) for number in re.findall(r'\d+', text)]
    if not numbers:
        return None, None
    if len(numbers) == 1:
        return numbers[0], numbers[0]
    return numbers[0], numbers[1]


def extract_email(text: str) -> str:
    '''Izvelk e-pasta adresi no teksta, ja t?da ir.'''
    match = re.search(r'[\w.+-]+@[\w.-]+\.\w+', text, flags=re.UNICODE)
    return match.group(0) if match else ''


def extract_phone(text: str) -> str:
    '''Izvelk Latvijai rakstur?gu t?lru?a numuru no teksta, ja t?ds ir.'''
    match = re.search(r'\+371\s?\d{8}', text)
    return match.group(0) if match else ''


def employment_type(text: str) -> str:
    '''Atgrie? kompaktu nodarbin?t?bas veida eti?eti no nodarbin?t?bas atbildes.'''
    lowered = text.lower()
    if 'nepilnu slodzi' in lowered:
        return 'nepilna slodze'
    if 'pilnu slodzi' in lowered:
        return 'pilna slodze'
    if 'pension?r' in lowered:
        return 'pension?rs/pension?re'
    return ''


def occupation(text: str) -> str:
    '''Izvelk profesijas fr?zi no nodarbin?t?bas atbildes.'''
    match = re.search(r'\bpar (.+?)(?=,|\sun\b|$)', text, flags=re.IGNORECASE | re.UNICODE)
    if match:
        return compact_value(match.group(1))

    lowered = text.lower()
    if 'pension?re' in lowered:
        return 'pension?re'
    if 'pension?rs' in lowered:
        return 'pension?rs'
    return ''


In [ ]:
def survey_row(path: Path, stopwords: set[str]) -> dict[str, str | int | None]:
    '''Pars? vienu aptaujas teksta failu plakan? CSV rind?.'''
    answers = parse_survey_blocks(read_text(path))
    parsed_answers = {
        field_name: answers.get(question_number, '')
        for question_number, field_name in QUESTION_FIELD_MAP.items()
    }

    row: dict[str, str | int | None] = {column: '' for column in CSV_COLUMNS}
    row['source_file'] = path.name

    row['name'] = compact_value(str(parsed_answers['name']))
    row['age_answer'] = str(parsed_answers['age_answer'])
    row['gender'] = compact_value(str(parsed_answers['gender']))
    row['employment_answer'] = str(parsed_answers['employment_answer'])
    row['education_answer'] = str(parsed_answers['education_answer'])
    row['household_answer'] = str(parsed_answers['household_answer'])
    row['income_answer'] = str(parsed_answers['income_answer'])
    row['economic_pressures_answer'] = str(parsed_answers['economic_pressures_answer'])
    row['inflation_impact_answer'] = str(parsed_answers['inflation_impact_answer'])
    row['job_finance_security_answer'] = str(parsed_answers['job_finance_security_answer'])
    row['riga_economy_view_answer'] = str(parsed_answers['riga_economy_view_answer'])
    row['improvements_answer'] = str(parsed_answers['improvements_answer'])

    row['age'] = first_int(str(parsed_answers['age_answer']))
    row['email'] = extract_email(str(parsed_answers['contact_answer']))
    row['phone'] = extract_phone(str(parsed_answers['contact_answer']))
    row['employment_type'] = employment_type(str(parsed_answers['employment_answer']))
    row['occupation'] = occupation(str(parsed_answers['employment_answer']))
    row['years_in_profession'] = first_int(str(parsed_answers['employment_answer']))
    income_min, income_max = income_range(str(parsed_answers['income_answer']))
    row['income_min_eur'] = income_min
    row['income_max_eur'] = income_max
    row['drive_km_week'] = first_int(str(parsed_answers['drive_answer']))
    row['bike_km_week'] = first_int(str(parsed_answers['bike_answer']))

    for field_name in TEXT_FIELDS_FOR_KEYWORDS:
        row[f'{field_name}_keywords'] = keyword_text(str(row[field_name]), stopwords)

    return row


def write_csv(
    path: Path,
    rows: list[dict[str, str | int | None]],
    encoding: str = 'utf-8',
) -> None:
    '''Saglab? aptaujas rindas CSV fail?.'''
    with path.open('w', encoding=encoding, newline='') as file:
        writer = csv.DictWriter(file, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
def survey_folder_to_csv(
    input_folder: Path = SURVEYS_FOLDER,
    output_path: Path | None = None,
    stopwords_path: Path = STOPWORDS_PATH,
) -> list[dict[str, str | int | None]]:
    '''Izpilda aptaujas mapes uz CSV pl?smu ar notebook draudz?giem noklus?jumiem.'''
    input_folder = Path(input_folder)
    resolved_output_path = Path(output_path) if output_path else input_folder / DEFAULT_OUTPUT_NAME

    if not input_folder.exists() or not input_folder.is_dir():
        raise FileNotFoundError(f'Ievades mape nav atrasta: {input_folder}')

    stopwords = load_stopwords(stopwords_path)
    survey_paths = sorted(input_folder.glob('*.txt'))
    rows = [survey_row(path, stopwords) for path in survey_paths]

    write_csv(resolved_output_path, rows)

    print(f'Ievades mape: {input_folder}')
    print(f'Apstr?d?to aptaujas failu skaits: {len(rows)}')
    print(f'CSV saglab?ts: {resolved_output_path}')
    return rows


survey_rows = survey_folder_to_csv()
survey_rows[:2]